# 进阶教程（十）：RAG 进阶检索三件套

> 基础向量检索之上，三个"提高召回质量"的进阶检索器：
> **自查询（SelfQuery）**、**父子文档（ParentDocument）**、**HyDE**。

## 本讲内容
1. 准备带元数据的向量库
2. SelfQuery（手写实现）：LLM 提取元数据过滤条件
3. ParentDocumentRetriever：小块检索 + 大块喂给 LLM
4. HyDE：先造"假设答案"再检索
5. 三者对比与选型

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-classic`、`langchain-deepseek`
- 检索章节使用本地 `BAAI/bge-small-zh-v1.5` 嵌入（首次运行会下载模型，约 100MB）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `advanced_tutorial/01-06` 的开发者
- 各讲结尾 FAQ 汇总了基于 langchain 1.3.14 实测的导入路径与坑

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. 准备向量库（带元数据）

三个进阶检索器都依赖**文档元数据**。先建一个带 `year` / `topic` 字段的 mini 库：

In [2]:

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

docs = [
    Document(page_content="2023 年公司推行弹性工作制，员工每周可远程办公两天。",
             metadata={"year": 2023, "topic": "制度"}),
    Document(page_content="2024 年新规：年假由 5 天调整为 10 天，需提前一周申请。",
             metadata={"year": 2024, "topic": "制度"}),
    Document(page_content="2023 年上线内部 AI 助手，用于代码补全与文档问答。",
             metadata={"year": 2023, "topic": "技术"}),
    Document(page_content="2024 年技术栈升级到 LangGraph，重构了所有 Agent 服务。",
             metadata={"year": 2024, "topic": "技术"}),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="adv10")
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("向量库就绪，文档数:", len(docs))

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

向量库就绪，文档数: 4


## 2. SelfQuery：让 LLM 读元数据（手写实现）

普通向量检索只比语义，无法理解"**2024 年**的制度有哪些"这种**约束条件**。
SelfQuery 的本质：让 LLM 把问题拆成「语义查询 + 元数据过滤」，先过滤再检索。

> 实测发现：`langchain_classic` 的 `SelfQueryRetriever.from_llm` 在当前环境
> （classic 1.0.8 + community 0.4.2）有版本不兼容 bug（内部硬编码导入已移除的
> `DatabricksVectorSearch`）。本讲**手写实现** SelfQuery 核心逻辑，原理相同且更可控。

In [ ]:

from pydantic import BaseModel, Field

# 1) 定义过滤条件结构：LLM 从问题里提取哪些字段有约束
class FilterSpec(BaseModel):
    year: int | None = Field(default=None, description="年份过滤，没有则 null")
    topic: str | None = Field(default=None, description="主题过滤，没有则 null")

parser = model.with_structured_output(FilterSpec)

# 2) LLM 提取过滤条件
question = "2024 年发布了哪些新制度？"
spec = parser.invoke(question) or FilterSpec()   # 结构化输出偶发 None，兜底为无过滤
print(f"LLM 提取的过滤条件: year={spec.year}, topic={spec.topic}")

# 3) 把提取到的条件翻译成 Chroma 的 filter（元数据过滤）
filters = {}
if spec.year is not None:
    filters["year"] = {"$eq": spec.year}
if spec.topic is not None:
    filters["topic"] = {"$eq": spec.topic}

# 4) 先过滤再检索
res = vectorstore.similarity_search(question, k=4, filter=filters or None)
print("SelfQuery（手写）命中:")
for d in res:
    print(f"  [{d.metadata['year']}] [{d.metadata['topic']}] {d.page_content[:30]}")

**要点**：手写 SelfQuery 四步 = 结构化提取 → 翻译过滤 → 向量检索。
`with_structured_output` 保证提取结果类型安全（year 是 int、topic 是 str），
这一步正是原版 SelfQuery 内部在做的事。

> 注意：DeepSeek 结构化输出**偶发返回 None**（模型未触发工具调用），
> 且字段提取存在语义随机性（如"新制度"可能被当成 topic）。生产务必加 `or FilterSpec()` 兜底。

## 3. ParentDocumentRetriever：小块检索、大块喂 LLM

矛盾：chunk 太小 → 检索精准但上下文破碎；chunk 太大 → 上下文完整但检索跑偏。
父子文档两全：**用小块建索引做检索，命中后返回它所属的大块**给 LLM。

> 实测导入路径：`ParentDocumentRetriever` 在 `langchain_classic.retrievers`；
> 它的 `docstore` 参数需要 `langchain_core.stores` 的 Store（**不是** `langgraph.store` 的）。

In [ ]:

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore   # 注意：是 langchain_core，不是 langgraph

# 父块（大，喂给 LLM）：800 字
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
# 子块（小，建索引）：200 字
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)

store = InMemoryStore()   # 存父块（生产可换持久化 store）

pdr = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 重新入库：自动切成父子两层（演示用一个新的 collection 避免和上面冲突）
big_docs = [Document(page_content="（这里放一长段正文，超过 800 字会被切成多个父块，"
                     "每个父块再切成若干 200 字子块建索引）" * 30)]
pdr.add_documents(big_docs)

# 检索：返回的是父块（上下文完整）
res = pdr.invoke("弹性工作制")
print("ParentDocument 命中父块数:", len(res))
print("首块长度:", len(res[0].page_content), "字符")

**要点**：检索走子块（精准），返回走父块（完整）。
`docstore` 负责存父子映射，`InMemoryStore` 仅演示，生产换持久化实现。

## 4. HyDE：先造"假设答案"再检索

用户提问与文档措辞常常不一致。HyDE（Hypothetical Document Embeddings）
先让 LLM 生成一个"假设性答案文档"，再用这个答案去向量检索——
因为答案的措辞风格更接近文档，召回率更高。

> 实测导入路径：`HypotheticalDocumentEmbedder` 在 `langchain_classic.chains`。

In [ ]:

from langchain_classic.chains import HypotheticalDocumentEmbedder
from langchain_core.prompts import ChatPromptTemplate

# 生成假设答案的链：让 LLM 假想文档会怎么写
hyde_prompt = ChatPromptTemplate.from_template(
    "请用一段话，假装你是公司制度文档，回答这个问题：{question}")
hyde_llm_chain = hyde_prompt | model

hyde = HypotheticalDocumentEmbedder(
    base_embeddings=embeddings,
    llm_chain=hyde_llm_chain,
)

# HyDE 检索：先造假设答案 → 再向量检索
from langchain_core.vectorstores import VectorStoreRetriever
query = "远程办公有什么规定？"
hypo_doc = hyde.embed_query(query)   # 内部先 LLM 生成假设答案再嵌入
print("假设答案已生成（用于检索的向量已就绪，长度:", len(hypo_doc), ")")

**要点**：HyDE 每次检索**多一次 LLM 调用**（生成假设答案），
所以适合"查询与文档措辞差异大、且对召回要求高"的场景，对延迟敏感的场景慎用。

## 5. 三件套对比与选型

| 检索器 | 解决什么 | 代价 | 适用 |
|---|---|---|---|
| SelfQuery | 元数据约束（时间/类别/数值） | 一次 LLM 调用 | 文档有结构化元数据 |
| ParentDocument | chunk 大小矛盾 | 双层存储 | 长文档、需要完整上下文 |
| HyDE | query 与 doc 措辞差异 | 一次 LLM 调用 | 口语化提问、跨领域检索 |

## 6. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| `SelfQueryRetriever.from_llm` 报 `ImportError: DatabricksVectorSearch` | classic 与 community 版本不兼容 | 用本讲手写实现（结构化提取 + Chroma filter） |
| SelfQuery 过滤不生效 | 向量库不支持元数据过滤 | Chroma 的 `filter` 参数支持 `{"$eq": ...}` 等操作符 |
| `with_structured_output` 返回 None | DeepSeek 偶发未触发工具调用 | `parser.invoke(...) or FilterSpec()` 兜底 |
| ParentDocument 报 `docstore should be BaseStore` | 用了 langgraph 的 InMemoryStore | 换 `langchain_core.stores.InMemoryStore` |
| ParentDocument 检索为空 | 父子映射没建好 | 用 `add_documents` 入库 |
| HyDE 结果反而更差 | 假设答案与真实文档风格差异大 | 只在口语化查询场景用，先 A/B 验证 |
| 三者能组合吗 | 可以 | 先 SelfQuery 过滤，再 ParentDocument 检索，HyDE 做查询改写 |